In [1]:
import sys
import os

notebook_path = os.getcwd() 
parent_dir = os.path.dirname(notebook_path)
project_root = os.path.dirname(parent_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from copy import deepcopy

In [3]:
import state_NN_models
import Filters
import utils
import Systems
from utils import losses, trainer, utils
from torch.utils.data import TensorDataset, DataLoader, random_split
from state_NN_models.StateBayesianKalmanNet import StateBayesianKalmanNet
from state_NN_models.StateKalmanNet import StateKalmanNet
from state_NN_models.StateKalmanNetWithKnownR import StateKalmanNetWithKnownR

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Used device: {device}")

Používané zařízení: cuda


In [5]:
state_dim_2d = 2
obs_dim_2d = 2

F_base_2d = torch.tensor([[1.0, 1.0], 
                          [0.0, 1.0]])

svd_F = torch.linalg.svd(F_base_2d)
F_true_2d = F_base_2d / svd_F.S[0]

H_true_2d = torch.eye(obs_dim_2d)

Q_true_2d = torch.eye(state_dim_2d) * 0.5 # Process noise
R_true_2d = torch.eye(obs_dim_2d) * 0.1 # Measurement noise

# Initial conditions
Ex0_true_2d = torch.tensor([[1.0], [0.0]])
P0_true_2d = torch.eye(state_dim_2d) * 1.5
F_model_2d = F_true_2d
H_model_2d = H_true_2d
Q_model_2d = torch.eye(state_dim_2d) * 0.05
R_model_2d = R_true_2d*4.0
Ex0_model_2d = torch.tensor([[0.5], [0.5]])
P0_model_2d = torch.eye(state_dim_2d) * 1.0

print("\nInitializing 2D Linear_Canonical system (replicating authors)...")
sys_true = Systems.DynamicSystem(
    state_dim=state_dim_2d, obs_dim=obs_dim_2d,
    Ex0=Ex0_true_2d, P0=P0_true_2d,
    Q=Q_true_2d, R=R_true_2d,
    F=F_true_2d, H=H_true_2d,
    device=device
)
sys_model = Systems.DynamicSystem(
    state_dim=state_dim_2d, obs_dim=obs_dim_2d,
    Ex0=Ex0_model_2d, P0=P0_model_2d,
    Q=Q_model_2d, R=R_model_2d,
    F=F_model_2d, H=H_model_2d,
    device=device
)
print("... 2D system initialized.")
sys_model_known_init = Systems.DynamicSystem(
    state_dim=state_dim_2d, obs_dim=obs_dim_2d,
    Ex0=Ex0_true_2d, P0=P0_true_2d,
    Q=Q_model_2d, R=R_model_2d,
    F=F_model_2d, H=H_model_2d,
    device=device
)
print("... 2D system initialized.")


Inicializuji 2D Linear_Canonical systém (replikace autorů)...
... 2D systém inicializován.
... 2D systém inicializován.


In [8]:
import torch
import torch.nn.functional as F
import numpy as np
from torch.utils.data import TensorDataset, DataLoader

# ==============================================================================
# 1. KONFIGURACE TESTU
# ==============================================================================
TEST_SEQ_LEN = 300
NUM_TEST_TRAJ = 2
J_SAMPLES_TEST = 25

# ==============================================================================
# 2. DATA PREPARATION
# ==============================================================================
print(f"\nGenerating {NUM_TEST_TRAJ} test trajectories of length {TEST_SEQ_LEN}...")
x_test, y_test = utils.generate_data(sys_true, num_trajectories=NUM_TEST_TRAJ, seq_len=TEST_SEQ_LEN)
test_dataset = TensorDataset(x_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
print("Data generation completed.")


# ==============================================================================
# 3. INITIALIZATION OF ALL FILTERS FOR COMPARISON
# ==============================================================================
ekf_mismatched = Filters.ExtendedKalmanFilter(sys_model)
ekf_ideal = Filters.ExtendedKalmanFilter(sys_true)
ukf_mismatched = Filters.UnscentedKalmanFilter(sys_model)
ukf_ideal = Filters.UnscentedKalmanFilter(sys_true)
aekf_mismatched = Filters.AdaptiveExtendedKalmanFilter(sys_model, Q_init=sys_model.Q, R_init=sys_model.R, alpha=0.985)
kf_ideal = Filters.KalmanFilter(sys_true)

akf_mismatched = Filters.AdaptiveKalmanFilter(sys_model,mdm_L=3,mdm_version=2)
akf_ideal = Filters.AdaptiveKalmanFilter(sys_true,mdm_L=3,mdm_version=2)

# pf_ideal = Filters.ParticleFilter(sys_true, num_particles=1000)
# pf_mismatched = Filters.ParticleFilter(sys_model_known_init, num_particles=1000)

pf_sir_ideal = Filters.ParticleFilterSIR(sys_true, num_particles=1000)
pf_sir_mismatched = Filters.ParticleFilterSIR(sys_model, num_particles=1000)

apf_ideal = Filters.AuxiliaryParticleFilter(sys_true, num_particles=1000)
apf_mismatched = Filters.AuxiliaryParticleFilter(sys_model, num_particles=1000)

pf_mh_ideal = Filters.ParticleFilterMH(sys_true, num_particles=1000)
pf_mh_mismatched = Filters.ParticleFilterMH(sys_model, num_particles=1000)

print("All model-based filters (EKF, UKF, AEKF) initialized.")


# ==============================================================================
# 4. EVALUATION LOOP
# ==============================================================================
# Lists for storing results from each trajectory
all_x_true_cpu = []
all_x_hat_bkn_cpu, all_P_hat_bkn_cpu = [], []
all_x_hat_classic_knet_cpu = []
all_x_hat_knetR_cpu, all_P_hat_knetR_cpu = [], []
all_x_hat_ekf_mismatched_cpu, all_P_hat_ekf_mismatched_cpu = [], []
all_x_hat_ekf_ideal_cpu, all_P_hat_ekf_ideal_cpu = [], []
all_x_hat_ukf_mismatched_cpu, all_P_hat_ukf_mismatched_cpu = [], []
all_x_hat_ukf_ideal_cpu, all_P_hat_ukf_ideal_cpu = [], []
all_x_hat_aekf_mismatched_cpu, all_P_hat_aekf_mismatched_cpu = [], []
all_x_hat_kf_ideal_cpu, all_P_hat_kf_ideal_cpu = [], []
all_x_hat_pf_ideal_cpu, all_P_hat_pf_ideal_cpu = [], []
all_x_hat_pf_mismatched_cpu, all_P_hat_pf_mismatched_cpu = [], []
all_x_hat_pf_sir_ideal_cpu, all_P_hat_pf_sir_ideal_cpu = [], []
all_x_hat_pf_sir_mismatched_cpu, all_P_hat_pf_sir_mismatched_cpu = [], []
all_x_hat_apf_ideal_cpu, all_P_hat_apf_ideal_cpu = [], []
all_x_hat_apf_mismatched_cpu, all_P_hat_apf_mismatched_cpu = [], []
all_x_hat_pf_mh_ideal_cpu, all_P_hat_pf_mh_ideal_cpu = [], []
all_x_hat_pf_mh_mismatched_cpu, all_P_hat_pf_mh_mismatched_cpu = [], []
all_x_hat_akf_ideal_cpu, all_P_hat_akf_ideal_cpu = [], []
all_x_hat_akf_mismatched_cpu, all_P_hat_akf_mismatched_cpu = [], []


print(f"\nEvaluating models on {NUM_TEST_TRAJ} test trajectories...")


with torch.no_grad():
    for i, (x_true_seq_batch, y_test_seq_batch) in enumerate(test_loader):
        y_test_seq_gpu = y_test_seq_batch.squeeze(0).to(device)
        x_true_seq_gpu = x_true_seq_batch.squeeze(0).to(device)
        initial_state = x_true_seq_gpu[0, :].unsqueeze(0)
        
      # --- D. EKF (inaccurate and ideal) ---
        ekf_m_res = ekf_mismatched.process_sequence(y_test_seq_gpu, Ex0=sys_model.Ex0, P0=sys_model.P0)

        full_x_hat_ekf_m = ekf_m_res['x_filtered'] # The result is already a complete trajectory
        full_P_hat_ekf_m = ekf_m_res['P_filtered'] # The same for covariance

        ekf_i_res = ekf_ideal.process_sequence(y_test_seq_gpu, Ex0=sys_true.Ex0, P0=sys_true.P0)

        full_x_hat_ekf_i = ekf_i_res['x_filtered']
        full_P_hat_ekf_i = ekf_i_res['P_filtered']

        # --- E. UKF (inaccurate and ideal) ---
        ukf_m_res = ukf_mismatched.process_sequence(y_test_seq_gpu, Ex0=sys_model.Ex0, P0=sys_model.P0)

        full_x_hat_ukf_m = ukf_m_res['x_filtered']
        full_P_hat_ukf_m = ukf_m_res['P_filtered']

        ukf_i_res = ukf_ideal.process_sequence(y_test_seq_gpu, Ex0=sys_true.Ex0, P0=sys_true.P0)

        full_x_hat_ukf_i = ukf_i_res['x_filtered']
        full_P_hat_ukf_i = ukf_i_res['P_filtered']

        kf_i_res = kf_ideal.process_sequence(y_test_seq_gpu, Ex0=sys_true.Ex0, P0=sys_true.P0)

        full_x_hat_kf_i = kf_i_res['x_filtered']
        full_P_hat_kf_i = kf_i_res['P_filtered']



        apf_i_res = apf_ideal.process_sequence(y_test_seq_gpu, Ex0=sys_true.Ex0,P0=sys_true.P0)
        full_x_hat_apf_i = apf_i_res['x_filtered']
        full_P_hat_apf_i = apf_i_res['P_filtered']
        print(f"APF (ideal model) completed for trajectory {i + 1}/{NUM_TEST_TRAJ}.")

        apf_m_res = apf_mismatched.process_sequence(y_test_seq_gpu, Ex0=sys_model.Ex0, P0=sys_model.P0)
        full_x_hat_apf_m = apf_m_res['x_filtered']
        full_P_hat_apf_m = apf_m_res['P_filtered']
        print(f"APF (inaccurate model) completed for trajectory {i + 1}/{NUM_TEST_TRAJ}.")



        pf_sir_i_res = pf_sir_ideal.process_sequence(y_test_seq_gpu, Ex0=sys_true.Ex0,P0=sys_true.P0)
        full_x_hat_pf_sir_i = pf_sir_i_res['x_filtered']
        full_P_hat_pf_sir_i = pf_sir_i_res['P_filtered']
        full_particles_history_pf_sir_i = pf_sir_i_res['particles_history']
        print(f"PF-SIR (ideal model) completed for trajectory {i + 1}/{NUM_TEST_TRAJ}.")

        pf_sir_m_res = pf_sir_mismatched.process_sequence(y_test_seq_gpu, Ex0=sys_model.Ex0, P0=sys_model.P0)
        full_x_hat_pf_sir_m = pf_sir_m_res['x_filtered']
        full_P_hat_pf_sir_m = pf_sir_m_res['P_filtered']
        full_particles_history_pf_sir_m = pf_sir_m_res['particles_history']
        print(f"PF-SIR (inaccurate model) completed for trajectory {i + 1}/{NUM_TEST_TRAJ}.")

        pf_mh_i_res = pf_mh_ideal.process_sequence(y_test_seq_gpu, Ex0=sys_true.Ex0,P0=sys_true.P0)
        full_x_hat_pf_mh_i = pf_mh_i_res['x_filtered']
        full_P_hat_pf_mh_i = pf_mh_i_res['P_filtered']
        print(f"PF-MH (ideal model) completed for trajectory {i + 1}/{NUM_TEST_TRAJ}.")

        pf_mh_m_res = pf_mh_mismatched.process_sequence(y_test_seq_gpu, Ex0=sys_model.Ex0, P0=sys_model.P0)
        full_x_hat_pf_mh_m = pf_mh_m_res['x_filtered']
        full_P_hat_pf_mh_m = pf_mh_m_res['P_filtered']
        print(f"PF-MH (inaccurate model) completed for trajectory {i + 1}/{NUM_TEST_TRAJ}.")

        akf_i_res,_,_ = akf_ideal.process_sequence_adaptively(y_test_seq_gpu)
        full_x_hat_akf_i = akf_i_res['x_filtered']
        full_P_hat_akf_i = akf_i_res['P_filtered']

        akf_m_res,_,_ = akf_mismatched.process_sequence_adaptively(y_test_seq_gpu)
        full_x_hat_akf_m = akf_m_res['x_filtered']
        full_P_hat_akf_m = akf_m_res['P_filtered']

        # --- G. Storing all results on CPU ---
        all_x_true_cpu.append(x_true_seq_gpu.cpu())
        all_x_hat_ekf_mismatched_cpu.append(full_x_hat_ekf_m.cpu()); all_P_hat_ekf_mismatched_cpu.append(full_P_hat_ekf_m.cpu())
        all_x_hat_ekf_ideal_cpu.append(full_x_hat_ekf_i.cpu()); all_P_hat_ekf_ideal_cpu.append(full_P_hat_ekf_i.cpu())
        all_x_hat_ukf_mismatched_cpu.append(full_x_hat_ukf_m.cpu()); all_P_hat_ukf_mismatched_cpu.append(full_P_hat_ukf_m.cpu())
        all_x_hat_ukf_ideal_cpu.append(full_x_hat_ukf_i.cpu()); all_P_hat_ukf_ideal_cpu.append(full_P_hat_ukf_i.cpu())
        all_x_hat_kf_ideal_cpu.append(full_x_hat_kf_i.cpu()); all_P_hat_kf_ideal_cpu.append(full_P_hat_kf_i.cpu())
        # all_x_hat_pf_ideal_cpu.append(full_x_hat_pf_i.cpu()); all_P_hat_pf_ideal_cpu.append(full_P_hat_pf_i.cpu())
        # all_x_hat_pf_mismatched_cpu.append(full_x_hat_pf_m.cpu()); all_P_hat_pf_mismatched_cpu.append(full_P_hat_pf_m.cpu())
        all_x_hat_pf_sir_ideal_cpu.append(full_x_hat_pf_sir_i.cpu()); all_P_hat_pf_sir_ideal_cpu.append(full_P_hat_pf_sir_i.cpu())
        all_x_hat_pf_sir_mismatched_cpu.append(full_x_hat_pf_sir_m.cpu()); all_P_hat_pf_sir_mismatched_cpu.append(full_P_hat_pf_sir_m.cpu())
        all_x_hat_apf_ideal_cpu.append(full_x_hat_apf_i.cpu()); all_P_hat_apf_ideal_cpu.append(full_P_hat_apf_i.cpu())
        all_x_hat_apf_mismatched_cpu.append(full_x_hat_apf_m.cpu()); all_P_hat_apf_mismatched_cpu.append(full_P_hat_apf_m.cpu())
        all_x_hat_pf_mh_ideal_cpu.append(full_x_hat_pf_mh_i.cpu()); all_P_hat_pf_mh_ideal_cpu.append(full_P_hat_pf_mh_i.cpu())
        all_x_hat_pf_mh_mismatched_cpu.append(full_x_hat_pf_mh_m.cpu()); all_P_hat_pf_mh_mismatched_cpu.append(full_P_hat_pf_mh_m.cpu())
        all_x_hat_akf_ideal_cpu.append(full_x_hat_akf_i.cpu()); all_P_hat_akf_ideal_cpu.append(full_P_hat_akf_i.cpu())
        all_x_hat_akf_mismatched_cpu.append(full_x_hat_akf_m.cpu()); all_P_hat_akf_mismatched_cpu.append(full_P_hat_akf_m.cpu())

        print(f"Completed trajectory {i + 1}/{NUM_TEST_TRAJ}...")

# ==============================================================================
# 5. FINAL CALCULATION AND OUTPUT OF METRICS
# ==============================================================================
# Lists for collecting metrics
mse_bkn, anees_bkn = [], []; mse_classic_knet = []; mse_knetR, anees_knetR = [], []
mse_ekf_mis, anees_ekf_mis = [], []; mse_ekf_ideal, anees_ekf_ideal = [], []
mse_ukf_mis, anees_ukf_mis = [], []; mse_ukf_ideal, anees_ukf_ideal = [], []
mse_aekf_mis, anees_aekf_mis = [], []; mse_kf_ideal, anees_kf_ideal = [], []
mse_pf_ideal, anees_pf_ideal = [], []; mse_pf_mis, anees_pf_mis = [], []
mse_pf_sir_ideal, anees_pf_sir_ideal = [], []; mse_pf_sir_mis, anees_pf_sir_mis = [], []
mse_apf_ideal, anees_apf_ideal = [], []; mse_apf_mis, anees_apf_mis = [], []
all_particles_history_pf_sir_mismatched_cpu = []
all_particles_history_pf_sir_ideal_cpu = []
mse_pf_mh_ideal, anees_pf_mh_ideal = [], []; mse_pf_mh_mis, anees_pf_mh_mis = [], []
mse_akf_ideal, anees_akf_ideal = [], []; mse_akf_mis, anees_akf_mis = [], []

print("\nCalculating final metrics for individual trajectories...")

with torch.no_grad():
    for i in range(NUM_TEST_TRAJ):
        x_true = all_x_true_cpu[i]
        def get_metrics(x_hat, P_hat):
            # Length of true trajectory
            true_len = x_true.shape[0]
            
            # Trim estimates to have the same length as true data.
            # We take the first `true_len` elements (i.e., for t=0 to t=299).
            x_hat_aligned = x_hat[:true_len]
            P_hat_aligned = P_hat[:true_len]
            
            # Now all tensors have the same length
            mse = F.mse_loss(x_hat_aligned, x_true).item()
            anees = utils.calculate_anees_vectorized(x_true.unsqueeze(0), x_hat_aligned.unsqueeze(0), P_hat_aligned.unsqueeze(0))
            return mse, anees

        # Calculations for all models
        mse, anees = get_metrics(all_x_hat_ekf_mismatched_cpu[i], all_P_hat_ekf_mismatched_cpu[i]); mse_ekf_mis.append(mse); anees_ekf_mis.append(anees)
        mse, anees = get_metrics(all_x_hat_ekf_ideal_cpu[i], all_P_hat_ekf_ideal_cpu[i]); mse_ekf_ideal.append(mse); anees_ekf_ideal.append(anees)
        mse, anees = get_metrics(all_x_hat_ukf_mismatched_cpu[i], all_P_hat_ukf_mismatched_cpu[i]); mse_ukf_mis.append(mse); anees_ukf_mis.append(anees)
        mse, anees = get_metrics(all_x_hat_ukf_ideal_cpu[i], all_P_hat_ukf_ideal_cpu[i]); mse_ukf_ideal.append(mse); anees_ukf_ideal.append(anees)
        mse, anees = get_metrics(all_x_hat_kf_ideal_cpu[i], all_P_hat_kf_ideal_cpu[i]); mse_kf_ideal.append(mse); anees_kf_ideal.append(anees)
        # mse, anees = get_metrics(all_x_hat_pf_ideal_cpu[i], all_P_hat_pf_ideal_cpu[i]); mse_pf_ideal.append(mse); anees_pf_ideal.append(anees)
        # mse, anees = get_metrics(all_x_hat_pf_mismatched_cpu[i], all_P_hat_pf_mismatched_cpu[i]); mse_pf_mis.append(mse); anees_pf_mis.append(anees)
        mse, anees = get_metrics(all_x_hat_pf_sir_ideal_cpu[i], all_P_hat_pf_sir_ideal_cpu[i]); mse_pf_sir_ideal.append(mse); anees_pf_sir_ideal.append(anees)
        mse, anees = get_metrics(all_x_hat_pf_sir_mismatched_cpu[i], all_P_hat_pf_sir_mismatched_cpu[i]); mse_pf_sir_mis.append(mse); anees_pf_sir_mis.append(anees)
        mse, anees = get_metrics(all_x_hat_apf_ideal_cpu[i], all_P_hat_apf_ideal_cpu[i]); mse_apf_ideal.append(mse); anees_apf_ideal.append(anees)
        mse, anees = get_metrics(all_x_hat_apf_mismatched_cpu[i], all_P_hat_apf_mismatched_cpu[i]); mse_apf_mis.append(mse); anees_apf_mis.append(anees)
        mse, anees = get_metrics(all_x_hat_pf_mh_ideal_cpu[i], all_P_hat_pf_mh_ideal_cpu[i]); mse_pf_mh_ideal.append(mse); anees_pf_mh_ideal.append(anees)
        mse, anees = get_metrics(all_x_hat_pf_mh_mismatched_cpu[i], all_P_hat_pf_mh_mismatched_cpu[i]); mse_pf_mh_mis.append(mse); anees_pf_mh_mis.append(anees)
        mse, anees = get_metrics(all_x_hat_akf_ideal_cpu[i], all_P_hat_akf_ideal_cpu[i]); mse_akf_ideal.append(mse); anees_akf_ideal.append(anees)
        mse, anees = get_metrics(all_x_hat_akf_mismatched_cpu[i], all_P_hat_akf_mismatched_cpu[i]); mse_akf_mis.append(mse); anees_akf_mis.append(anees)

# Function for safe averaging
def avg(metric_list): return np.mean([m for m in metric_list if not np.isnan(m)])
state_dim_for_nees = all_x_true_cpu[0].shape[1]

# --- Final table output ---
print("\n" + "="*80)
print(f"FINAL RESULTS (average over {NUM_TEST_TRAJ} runs)")
print("="*80)
print(f"{'Model':<35} | {'Average MSE':<20} | {'Average ANEES':<20}")
print("-" * 80)
print("-" * 80)
print(f"{'--- Model-Based Filters ---':<35} | {'':<20} | {'':<20}")
print(f"{'EKF (Inaccurate model)':<35} | {avg(mse_ekf_mis):<20.4f} | {avg(anees_ekf_mis):<20.4f}")
print(f"{'UKF (Inaccurate model)':<35} | {avg(mse_ukf_mis):<20.4f} | {avg(anees_ukf_mis):<20.4f}")
# print(f"{'PF (Inaccurate model)':<35} | {avg(mse_pf_mis):<20.4f} | {avg(anees_pf_mis):<20.4f}")
print(f"{'PF-SIR (Inaccurate model)':<35} | {avg(mse_pf_sir_mis):<20.4f} | {avg(anees_pf_sir_mis):<20.4f}")
print(f"{'APF (Inaccurate model)':<35} | {avg(mse_apf_mis):<20.4f} | {avg(anees_apf_mis):<20.4f}")
print(f"{'PF-MH (Inaccurate model)':<35} | {avg(mse_pf_mh_mis):<20.4f} | {avg(anees_pf_mh_mis):<20.4f}")
print(f"{'AKF (Mismatched model)':<35} | {avg(mse_akf_mis):<20.4f} | {avg(anees_akf_mis):<20.4f}")
print("-" * 80)
print(f"{'--- Benchmarks ---':<35} | {'':<20} | {'':<20}")
print(f"{'EKF (Ideal model)':<35} | {avg(mse_ekf_ideal):<20.4f} | {avg(anees_ekf_ideal):<20.4f}")
print(f"{'UKF (Ideal model)':<35} | {avg(mse_ukf_ideal):<20.4f} | {avg(anees_ukf_ideal):<20.4f}")
print(f"{'KF (Ideal model)':<35} | {avg(mse_kf_ideal):<20.4f} | {avg(anees_kf_ideal):<20.4f}")
# print(f"{'PF (Ideal model)':<35} | {avg(mse_pf_ideal):<20.4f} | {avg(anees_pf_ideal):<20.4f}")
print(f"{'PF-SIR (Ideal model)':<35} | {avg(mse_pf_sir_ideal):<20.4f} | {avg(anees_pf_sir_ideal):<20.4f}")
print(f"{'APF (Ideal model)':<35} | {avg(mse_apf_ideal):<20.4f} | {avg(anees_apf_ideal):<20.4f}")
print(f"{'PF-MH (Ideal model)':<35} | {avg(mse_pf_mh_ideal):<20.4f} | {avg(anees_pf_mh_ideal):<20.4f}")
print(f"{'AKF (Ideal model)':<35} | {avg(mse_akf_ideal):<20.4f} | {avg(anees_akf_ideal):<20.4f}")
print("="*80)


Generuji 2 testovacích trajektorií o délce 300...
Generování dat dokončeno.
Všechny model-based filtry (EKF, UKF, AEKF) inicializovány.

Vyhodnocuji modely na 2 testovacích trajektoriích...
APF (ideální model) dokončen pro trajektorii 1/2.
APF (nepřesný model) dokončen pro trajektorii 1/2.
PF-SIR (ideální model) dokončen pro trajektorii 1/2.
PF-SIR (nepřesný model) dokončen pro trajektorii 1/2.
PF-MH (ideální model) dokončen pro trajektorii 1/2.
PF-MH (nepřesný model) dokončen pro trajektorii 1/2.
Spouštím MDM odhad Q a R...
MDM odhad dokončen.
Odhadnuté Q:
 [[0.63083076 0.01765879]
 [0.01765879 0.47976938]]
Odhadnuté R:
 [[-0.00480343  0.0193079 ]
 [ 0.0193079   0.08151945]]
Matice Q a R v Kalmanově filtru byly aktualizovány.
Spouštím MDM odhad Q a R...
MDM odhad dokončen.
Odhadnuté Q:
 [[0.63083076 0.01765879]
 [0.01765879 0.47976938]]
Odhadnuté R:
 [[-0.00480343  0.0193079 ]
 [ 0.0193079   0.08151945]]
Matice Q a R v Kalmanově filtru byly aktualizovány.
Dokončena trajektorie 1/2...

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# --- Importy pro interaktivitu ---
import ipywidgets as widgets
from ipywidgets import interact, IntSlider
from IPython.display import display

# ==============================================================================
# 1. KONFIGURACE A INICIALIZACE
# ==============================================================================
print("Inicializuji PF-SIR filtry pro diagnostiku...")

# For diagnostics we will work with only one trajectory.
# Make sure that NUM_TEST_TRAJ=1 in the cell where you generate data.
if 'NUM_TEST_TRAJ' not in locals() or NUM_TEST_TRAJ > 1:
    if 'NUM_TEST_TRAJ' in locals():
        print(f"WARNING: NUM_TEST_TRAJ was {NUM_TEST_TRAJ}. For the purposes of this diagnostic only the first trajectory is used.")
    else:
        # If NUM_TEST_TRAJ does not exist, we set it
        NUM_TEST_TRAJ = 1

# Initialization of filters we want to test
# You can adjust the number of particles for experimentation
NUM_PARTICLES_DIAG = 200
pf_sir_ideal = Filters.ParticleFilterSIR(sys_true, num_particles=NUM_PARTICLES_DIAG)
pf_sir_mismatched = Filters.ParticleFilterSIR(sys_model_known_init, num_particles=NUM_PARTICLES_DIAG)

print("Initialization completed.")

# ==============================================================================
# 2. PROCESSING OF ONE TRAJECTORY
# ==============================================================================
print("\nProcessing the first test trajectory...")

# We take the first trajectory from our test data
x_true_seq_gpu = x_test[0].to(device)
y_test_seq_gpu = y_test[0].to(device)

with torch.no_grad():
    # --- RUN OF THE IDEAL MODEL ---
    # Make sure your process_sequence returns 'particles_history'
    pf_sir_i_res = pf_sir_ideal.process_sequence(
        y_test_seq_gpu, 
        Ex0=sys_true.Ex0,
        P0=sys_true.P0,
        Q=sys_true.Q,
        R=sys_true.R
    )
    print("PF-SIR (ideal model) completed.")

    # --- RUN OF THE INACCURATE MODEL ---
    pf_sir_m_res = pf_sir_mismatched.process_sequence(
        y_test_seq_gpu,
        Ex0=sys_model_known_init.Ex0,
        P0=sys_model_known_init.P0,
        Q=sys_model_known_init.Q,
        R=sys_model_known_init.R
    )
    print("PF-SIR (inaccurate model) completed.")

print("Trajectory processing completed.")

# ==============================================================================
# 3. INTERACTIVE VISUALIZATION FUNCTION
# ==============================================================================

def plot_particle_filter_step(x_true, y_meas, pf_results, t, title_prefix=""):
    """
    Plots the filter state at ONE specific time step 't'.
    """
    # We convert data to NumPy once at the start if they are not yet
    x_true_np = x_true if isinstance(x_true, np.ndarray) else x_true.cpu().numpy()
    y_meas_np = y_meas if isinstance(y_meas, np.ndarray) else y_meas.cpu().numpy()
    x_est_np = pf_results['x_filtered'].cpu().numpy()
    particles_history = pf_results.get('particles_history')
    
    # We determine the state dimension and axis indices
    state_dim = x_true_np.shape[1]
    x_axis_idx = 0
    y_axis_idx = 1
    if state_dim > 2 and state_dim % 2 == 0:
        y_axis_idx = state_dim // 2
        
    plt.figure(figsize=(11, 9))
    
    # We plot the ENTIRE history of trajectories and measurements for context
    plt.plot(x_true_np[:, x_axis_idx], x_true_np[:, y_axis_idx], 'g-', linewidth=2, alpha=0.3, label='Actual trajectory (entire)')
    plt.scatter(y_meas_np[:, 0], y_meas_np[:, 1], color='b', marker='x', s=15, alpha=0.3, label='Measurements (all)')
    plt.plot(x_est_np[:, x_axis_idx], x_est_np[:, y_axis_idx], 'r--', linewidth=1.5, alpha=0.3, label='Filter estimate (entire)')

    # We highlight the CURRENT positions at time 't'
    plt.plot(x_true_np[t, x_axis_idx], x_true_np[t, y_axis_idx], 'go', markersize=12, label=f'Actual at t={t}')
    if t < len(y_meas_np):
        plt.plot(y_meas_np[t, 0], y_meas_np[t, 1], 'bo', markersize=12, markerfacecolor='none', markeredgewidth=2, label=f'Measurement at t={t}')
    plt.plot(x_est_np[t, x_axis_idx], x_est_np[t, y_axis_idx], 'rs', markersize=8, label=f'Odhad v t={t}')
    
    # We plot the CURRENT particle cloud
    if particles_history and t < len(particles_history):
        particles = particles_history[t]
        sample_size = min(len(particles), 1000)
        sample_indices = np.random.choice(len(particles), sample_size, replace=False)
        
        plt.scatter(particles[sample_indices, x_axis_idx], particles[sample_indices, y_axis_idx], 
                    color='black', s=10, alpha=0.6, label=f'Particles at t={t}')

    plt.xlabel(f"Stav [{x_axis_idx}]")
    plt.ylabel(f"Stav [{y_axis_idx}]")
    plt.title(f"{title_prefix} - State at time t={t}", fontsize=16)
    plt.legend()
    plt.grid(True)
    plt.axis('equal')
    plt.show()

# ==============================================================================
# 4. LAUNCHING INTERACTIVE VISUALIZATION
# ==============================================================================
print("\nCreating interactive diagnostic plots...")
print("Use the sliders below to explore the evolution of filters over time.")

# We prepare data for passing to functions (just for clarity)
x_true_np_diag = x_true_seq_gpu.cpu().numpy()
y_meas_np_diag = y_test_seq_gpu.cpu().numpy()

# Function that will be called by the slider for the IDEAL model
def interactive_plot_ideal(t):
    # Zkontrolujeme, zda historie existuje
    if pf_sir_i_res.get('particles_history') is None:
        print("ERROR: 'particles_history' not found in ideal filter results. Modify the process_sequence method to return it.")
        return
    plot_particle_filter_step(x_true_np_diag, y_meas_np_diag, pf_sir_i_res, t, 
                              title_prefix="PF-SIR (Ideal model)")

"# Function that will be called by the slider for the INACCURATE model"
def interactive_plot_mismatched(t):
    # Zkontrolujeme, zda historie existuje
    if pf_sir_m_res.get('particles_history') is None:
        print("ERROR: 'particles_history' not found in inaccurate filter results. Modify the process_sequence method to return it.")
        return
    plot_particle_filter_step(x_true_np_diag, y_meas_np_diag, pf_sir_m_res, t, 
                                title_prefix="PF-SIR (Inaccurate model)")

"# We create sliders and connect them with functions"
seq_len = x_true_np_diag.shape[0]

print("\n--- Interactive graph for IDEAL model ---")
interact(interactive_plot_ideal, t=IntSlider(min=0, max=seq_len-1, step=1, value=0, description="Time step (t):", continuous_update=False))

print("\n--- Interactive graph for INACCURATE model ---")
interact(interactive_plot_mismatched, t=IntSlider(min=0, max=seq_len-1, step=1, value=0, description="Time step (t):", continuous_update=False))

Inicializuji PF-SIR filtry pro diagnostiku...
Inicializace dokončena.

Zpracovávám první testovací trajektorii...
PF-SIR (ideální model) dokončen.
PF-SIR (nepřesný model) dokončen.
Zpracování trajektorie dokončeno.

Vytvářím interaktivní diagnostické grafy...
Použijte slidery níže pro prozkoumání vývoje filtrů v čase.

--- Interaktivní graf pro IDEÁLNÍ model ---


interactive(children=(IntSlider(value=0, continuous_update=False, description='Časový krok (t):', max=99), Out…


--- Interaktivní graf pro NEPŘESNÝ model ---


interactive(children=(IntSlider(value=0, continuous_update=False, description='Časový krok (t):', max=99), Out…

<function __main__.interactive_plot_mismatched(t)>

In [ ]:
Q_history = aekf_m_res['Q_history']
R_history = aekf_m_res['R_history']

plt.figure(figsize=(20, 10))
plt.subplot(2, 1, 1)
plt.plot(Q_history[:,0,0].cpu(), label='Q[0,0]')
plt.plot(R_history[:,0,0].cpu(), label='R[0,0]')
plt.axhline(sys_true.Q[0,0].cpu().item(), color='blue', linestyle='--', label='True Q[0,0]')
plt.axhline(sys_true.R[0,0].cpu().item(), color='orange', linestyle='--', label='True R[0,0]')
plt.axhline(sys_model.Q[0,0].cpu().item(), color='blue', linestyle=':', label='Initial Q[0,0]')
plt.axhline(sys_model.R[0,0].cpu().item(), color='orange', linestyle=':', label='Initial R[0,0]')
plt.xlabel('Time Step')
plt.ylabel('Value')
plt.title('Adaptation of Q and R over Time')
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(20, 10))
plt.subplot(2, 1, 2)
plt.plot(Q_history[:,1,1].cpu(), label='Q[1,1]')
plt.plot(R_history[:,1,1].cpu(), label='R[1,1]')
plt.axhline(sys_true.Q[1,1].cpu().item(), color='blue', linestyle='--', label='True Q[1,1]')
plt.axhline(sys_true.R[1,1].cpu().item(), color='orange', linestyle='--', label='True R[1,1]')
plt.axhline(sys_model.Q[1,1].cpu().item(), color='blue', linestyle=':', label='Initial Q[1,1]')
plt.axhline(sys_model.R[1,1].cpu().item(), color='orange', linestyle=':', label='Initial R[1,1]')
plt.xlabel('Time Step')
plt.ylabel('Value')
plt.title('Adaptation of Q and R over Time')
plt.legend()
plt.grid()
plt.show()

NameError: name 'aekf_m_res' is not defined